
## 🚦 Tổng quan Hệ thống (Pipeline Overview)

Đoạn mã này cấu thành một (pipeline) hoàn chỉnh để phân tích hiệu suất giao thông tại một ngã tư 4 hướng bằng Computer Vision. Hệ thống được chia thành 4 giai đoạn chính, hoạt động nối tiếp nhau thông qua các giao diện người dùng tương tác (Gradio):

```text
[ VIDEO GIAO THÔNG ĐẦU VÀO ]
           │
           ▼
┌────────────────────────────────────────┐       [ DỮ LIỆU ĐẦU RA ]
│ Bước 1: CALIBRATION (Định chuẩn)       ├──────► calib_config.json
│ - Vẽ vùng ROI & Làn đường 4 hướng      │        (Chứa tọa độ pixel & kích thước thực)
│ - Tạo ma trận phối cảnh (Perspective)  │
└──────────────────┬─────────────────────┘
                   │
                   ▼
┌────────────────────────────────────────┐       [ DỮ LIỆU ĐẦU RA ]
│ Bước 2: DETECTION & TRACKING           ├──────► Dữ liệu: vehicle_tracks_xy.csv, vehicles.csv
│ - Object Detection (YOLO)              │        Video: vehicles-result.mp4
│ - Object Tracking (ByteTrack)          │
│ - Tính vận tốc (km/h) & Đếm xe         │
└──────────────────┬─────────────────────┘
                   │
                   ▼
┌────────────────────────────────────────┐       [ DỮ LIỆU ĐẦU RA ]
│ Bước 3: ANALYTICS (Phân tích)          ├──────► analytics.csv (Dữ liệu chuỗi thời gian)
│ - Tính toán Control Delay, Stopped     │        summary.json (Chỉ số tổng hợp)
│ - Xếp hạng Mức độ phục vụ (LOS A-F)    │
│ - Tính Lưu lượng thông qua (Throughput)│
└──────────────────┬─────────────────────┘
                   │
                   ▼
┌────────────────────────────────────────┐       [ KẾT QUẢ CUỐI CÙNG ]
│ Bước 4: VISUALIZATION (Trực quan hóa)  ├──────► Giao diện Dashboard (Gradio)
│ - Đọc dữ liệu từ Bước 3                │        (Hiển thị biểu đồ Lưu lượng, Tốc độ,
│ - Render biểu đồ bằng Matplotlib       │         Mức độ phục vụ - LOS theo từng làn)
└────────────────────────────────────────┘

```

### **Cell 1: Cài đặt môi trường (Environment Setup)**

* **Mục đích:** Tải và cài đặt các thư viện cần thiết để chạy toàn bộ pipeline.
* **Thành phần chính:**
* `ultralytics`: Sử dụng mô hình YOLO cho object detection (nhận diện phương tiện).
* `supervision`: Cung cấp các công cụ tracking (ByteTrack) và xử lý detections dễ dàng.
* `gradio`: Xây dựng giao diện Web UI tương tác trực tiếp trên Colab.
* `opencv-python-headless`: Xử lý hình ảnh và video.
* `numpy`, `pandas`: Xử lý mảng tính toán và cấu trúc dữ liệu bảng.

In [ ]:
# Cài thư viện
!pip install -q ultralytics supervision gradio opencv-python-headless numpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 11.5 MB/s eta 0:00:00



### **Cell 2: Calibration — Định chuẩn 4 hướng ngã tư**

* **Mục đích:** Khởi tạo một giao diện Web cho phép người dùng định nghĩa thủ công các Vùng quan tâm (ROI) và các làn đường (Lanes) cho 4 hướng (Bắc, Nam, Đông, Tây) từ một frame của video.
* **Workflow / Luồng hoạt động:**
  1. Tải video lên và hệ thống sẽ tự động trích xuất một vài frame đại diện để người dùng chọn.
  2. Người dùng chọn Hướng (vd: North) -> Chế độ ROI -> Click 4 điểm trên ảnh để tạo vùng ROI.
  3. Người dùng chuyển sang Chế độ Lane -> Click 4 điểm để vẽ từng làn đường bên trong ROI đó.
  4. Nhập kích thước thực tế (Chiều Rộng x Chiều Dài tính bằng mét) của vùng không gian vừa vẽ. Kích thước này rất quan trọng để hệ thống tạo ma trận biến đổi phối cảnh (Perspective Transform), giúp quy đổi từ pixel trên ảnh sang tọa độ thế giới thực (mét).
  5. Lặp lại cho đủ 4 hướng.


* **Đầu ra (Output):** Lưu toàn bộ cấu hình vào file `calib_config.json`.

In [ ]:
# ============================================================
# CELL 2 — Calibration: 4 hướng ROI cho ngã tư
# Workflow: chọn hướng → vẽ ROI → vẽ lanes → lưu → chọn hướng tiếp
# ============================================================

import cv2, numpy as np, gradio as gr, json, os
from pathlib import Path

DIRECTIONS  = ["North", "South", "East", "West"]
DIR_COLORS  = {"North":(0,255,0), "South":(0,100,255), "East":(255,200,0), "West":(255,0,200)}
LANE_COLORS = [(255,50,50),(50,255,50),(50,50,255),(255,255,50),(255,50,255),(50,165,255)]

state = {
    "video_path": None,
    "frame":      None,
    "directions": {d: {"roi_points":[], "real_w":10.0, "real_h":15.0,
                        "lanes":[], "current_lane":[]} for d in DIRECTIONS},
    "active_dir": "North",
    "mode":       "roi",
}

def S():  return state["directions"][state["active_dir"]]
def bgr2rgb(f): return cv2.cvtColor(f, cv2.COLOR_BGR2RGB)

def draw_overlay(frame):
    img = frame.copy()
    for d, data in state["directions"].items():
        c   = DIR_COLORS[d]
        pts = data["roi_points"]
        if len(pts) >= 2:
            for i in range(len(pts)-1): cv2.line(img, pts[i], pts[i+1], c, 2)
        if len(pts) == 4:
            cv2.line(img, pts[3], pts[0], c, 2)
            cx = sum(p[0] for p in pts)//4; cy = sum(p[1] for p in pts)//4
            cv2.putText(img, d, (cx-20, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c, 2)
        for li, lane in enumerate(data["lanes"]):
            cv2.polylines(img, [np.array(lane, np.int32)], True, LANE_COLORS[li%len(LANE_COLORS)], 1)

    # Active direction — highlight
    d, c, pts = state["active_dir"], DIR_COLORS[state["active_dir"]], S()["roi_points"]
    for i, (x,y) in enumerate(pts):
        cv2.circle(img, (x,y), 8, c, -1)
        cv2.putText(img, f"{d[0]}{i+1}", (x+8,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, c, 2)
    for li, lane in enumerate(S()["lanes"]):
        lc = LANE_COLORS[li%len(LANE_COLORS)]
        cv2.polylines(img, [np.array(lane, np.int32)], True, lc, 2)
        cx = sum(p[0] for p in lane)//4; cy = sum(p[1] for p in lane)//4
        cv2.putText(img, f"L{li+1}", (cx,cy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, lc, 2)
    for x,y in S()["current_lane"]:
        cv2.circle(img, (x,y), 6, (255,255,255), -1)

    hint = (f"[{d}] ROI {len(pts)}/4" if state["mode"]=="roi"
            else f"[{d}] Lane {len(S()['lanes'])+1}: {len(S()['current_lane'])}/4")
    cv2.putText(img, hint, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.putText(img, hint, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 1)
    return bgr2rgb(img)

# ── Handlers ──────────────────────────────────────────────────

def load_video(video_file):
    if not video_file: return None, "Chưa chọn video", gr.update(choices=[])
    state["video_path"] = video_file
    cap   = cv2.VideoCapture(video_file)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30
    idxs  = [0, total//4, total//2, 3*total//4]
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, f = cap.read()
        if ok: frames.append((idx, f))
    cap.release()
    if not frames: return None, "❌ Không đọc được frame", gr.update(choices=[])
    state["frame"] = frames[0][1]
    labels = [f"Frame {i} (~{i/fps:.1f}s)" for i,_ in frames]
    return draw_overlay(frames[0][1]), f"✅ {Path(video_file).name} | {total}f | {fps:.0f}fps", gr.update(choices=labels, value=labels[0])

def select_frame(choice, video_file):
    if not choice or not video_file: return None
    idx = int(choice.split()[1])
    cap = cv2.VideoCapture(video_file)
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, frame = cap.read(); cap.release()
    if ok: state["frame"] = frame; return draw_overlay(frame)

def set_direction(d):
    state["active_dir"] = d; state["mode"] = "roi"
    done = [x for x in DIRECTIONS if len(state["directions"][x]["roi_points"]) == 4]
    return draw_overlay(state["frame"]) if state["frame"] is not None else None, f"Hướng: {d} | Xong: {done}"

def set_mode(mode):
    state["mode"] = mode
    return draw_overlay(state["frame"]) if state["frame"] is not None else None, f"Mode: {mode}"

def on_click(evt: gr.SelectData):
    if state["frame"] is None: return None, "Upload video trước"
    x, y = int(evt.index[0]), int(evt.index[1])
    if state["mode"] == "roi":
        if len(S()["roi_points"]) < 4: S()["roi_points"].append((x,y))
        msg = f"[{state['active_dir']}] ROI {len(S()['roi_points'])}/4"
    else:
        S()["current_lane"].append((x,y))
        msg = f"[{state['active_dir']}] Lane {len(S()['lanes'])+1}: {len(S()['current_lane'])}/4"
        if len(S()["current_lane"]) == 4:
            S()["lanes"].append(S()["current_lane"].copy()); S()["current_lane"] = []
            msg = f"✅ Lane {len(S()['lanes'])} xong!"
    return draw_overlay(state["frame"]), msg

def clear_roi():
    S()["roi_points"] = []; state["mode"] = "roi"
    return draw_overlay(state["frame"]) if state["frame"] is not None else None, "Đã xóa ROI"

def clear_lane():
    if S()["lanes"]: S()["lanes"].pop()
    S()["current_lane"] = []
    return draw_overlay(state["frame"]) if state["frame"] is not None else None, "Đã xóa lane cuối"

def update_size(rw, rh):
    S()["real_w"] = float(rw); S()["real_h"] = float(rh)
    return f"[{state['active_dir']}] {rw}m × {rh}m đã lưu"

def save_all(weights, out_dir):
    missing_roi   = [d for d in DIRECTIONS if len(state["directions"][d]["roi_points"]) != 4]
    missing_lanes = [d for d in DIRECTIONS if not state["directions"][d]["lanes"]]
    if missing_roi:   return f"❌ Chưa vẽ ROI: {missing_roi}"
    if missing_lanes: return f"❌ Chưa vẽ lane: {missing_lanes}"
    os.makedirs(out_dir, exist_ok=True)
    config = {
        "video_path":   state["video_path"],
        "weights_path": weights or "yolo11l.pt",
        "output_dir":   out_dir,
        "directions":   {d: {"roi_points": v["roi_points"], "real_w_m": v["real_w"],
                              "real_h_m": v["real_h"], "lanes": v["lanes"]}
                         for d, v in state["directions"].items()},
    }
    path = os.path.join(out_dir, "calib_config.json")
    with open(path, "w") as f: json.dump(config, f, indent=2)
    total_lanes = sum(len(v["lanes"]) for v in state["directions"].values())
    return f"✅ Đã lưu! 4 hướng, {total_lanes} lanes\n{path}\n\n👉 Chạy Cell 3!"

# ── UI ────────────────────────────────────────────────────────

with gr.Blocks(title="Calibration — 4 hướng", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚦 Calibration — Ngã tư 4 hướng")
    gr.Markdown("**Workflow:** Chọn hướng → ROI (4 điểm) → Lane → Lặp 4 hướng → Lưu")

    with gr.Row():
        with gr.Column(scale=1):
            video_input    = gr.File(label="📹 Upload Video", file_types=["video"])
            frame_selector = gr.Radio(label="Chọn frame", choices=[], interactive=True)
            gr.Markdown("### 1️⃣ Hướng đang vẽ")
            dir_radio = gr.Radio(DIRECTIONS, value="North", label="")
            gr.Markdown("### 2️⃣ Mode")
            with gr.Row():
                btn_roi  = gr.Button("🟢 ROI",  variant="primary")
                btn_lane = gr.Button("🟠 Lane", variant="secondary")
            with gr.Row():
                btn_clr_roi  = gr.Button("🗑 Xóa ROI")
                btn_clr_lane = gr.Button("🗑 Xóa Lane cuối")
            gr.Markdown("### 3️⃣ Kích thước thực (hướng đang chọn)")
            real_w   = gr.Number(label="Rộng (m)", value=10.0, minimum=0.1)
            real_h   = gr.Number(label="Dài (m)",  value=15.0, minimum=0.1)
            btn_size = gr.Button("Cập nhật kích thước")
            gr.Markdown("### 💾 Lưu")
            weights_in = gr.Textbox(label="YOLO Weights", value="best.pt")
            out_dir_in = gr.Textbox(label="Output folder", value="/content/output")
            btn_save   = gr.Button("💾 Lưu tất cả 4 hướng", variant="primary", size="lg")
            status_box = gr.Textbox(label="Trạng thái", lines=5, interactive=False)

        with gr.Column(scale=2):
            canvas    = gr.Image(label="🖼️ Click để vẽ", interactive=True, height=540)
            click_msg = gr.Textbox(label="", interactive=False)

    video_input.change(load_video,     [video_input],              [canvas, status_box, frame_selector])
    frame_selector.change(select_frame,[frame_selector, video_input],[canvas])
    dir_radio.change(set_direction,    [dir_radio],                 [canvas, click_msg])
    btn_roi.click(lambda: set_mode("roi"),   outputs=[canvas, click_msg])
    btn_lane.click(lambda: set_mode("lane"), outputs=[canvas, click_msg])
    btn_clr_roi.click(clear_roi,   outputs=[canvas, click_msg])
    btn_clr_lane.click(clear_lane, outputs=[canvas, click_msg])
    btn_size.click(update_size, [real_w, real_h], [click_msg])
    canvas.select(on_click, outputs=[canvas, click_msg])
    btn_save.click(save_all, [weights_in, out_dir_in], [status_box])

demo.launch(share=True, quiet=True)

/tmp/ipykernel_518/290492635.py:141: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Calibration — 4 hướng", theme=gr.themes.Soft()) as demo:


* Running on public URL: https://b9924da70225516fa1.gradio.live


### **Cell 3: Detection & Tracking — Nhận diện & Theo dõi phương tiện**

* **Mục đích:** Xử lý video đầu vào dựa trên file cấu hình đã tạo ở Cell 2. Thực hiện nhận diện, gán ID theo dõi, tính tốc độ và đếm xe.
* **Quá trình xử lý logic:**
  * **Phối cảnh (Perspective):** Xây dựng ma trận `cv2.getPerspectiveTransform` cho từng hướng.
  * **Nhận diện & Tracking:** Chạy mô hình YOLO (chỉ lọc các class xe cộ: ô tô, xe máy, xe buýt, xe tải) và đẩy qua ByteTrack để duy trì ID (Tracking ID) cho từng xe qua các frame.
  * **Phân luồng & Tốc độ:** Kiểm tra xem tâm của xe (centroid) đang nằm trong ROI/Làn nào. Sử dụng sliding window (lịch sử tọa độ) và ma trận phối cảnh để tính quãng đường di chuyển thực tế (m) giữa các frame, từ đó suy ra vận tốc (km/h).
  * **Hiển thị (Overlay):** Vẽ bounding box xung quanh phương tiện, gán màu sắc theo làn đường, và hiển thị một bảng đếm xe (Count panel) trực tiếp trên góc video.


* **Đầu ra (Output):**
  * `vehicles-result.mp4`: Video đã được vẽ đè các thông số (bounding box, ID, tốc độ, vùng ROI).
  * `vehicle_tracks_xy.csv`: Bảng dữ liệu thô ghi lại tọa độ, tốc độ, làn đường của từng xe tại từng frame.
  * `vehicles.csv`: Bảng tóm tắt tốc độ trung bình/tối đa và tổng số frame xuất hiện của từng xe.
  * `lanes.csv`: Dữ liệu tọa độ các làn đường.

In [ ]:
import cv2, numpy as np, gradio as gr, json, os, csv, traceback
from pathlib import Path
from collections import defaultdict, deque
from ultralytics import YOLO
import supervision as sv

VEHICLE_CLASSES = {2:"car", 3:"motorcycle", 5:"bus", 7:"truck"}
LANE_COLORS_BGR = [(50,50,255),(50,255,50),(255,50,50),(50,255,255),(255,50,255),(50,165,255)]
DIR_COLORS_BGR  = {"North":(0,255,0),"South":(255,100,0),"East":(0,200,255),"West":(200,0,255)}

# ── Helpers ───────────────────────────────────────────────────

def build_perspective(roi_pts, real_w, real_h, px_per_m=100):
    src  = np.float32(roi_pts)
    ow, oh = int(real_w*px_per_m), int(real_h*px_per_m)
    dst  = np.float32([[0,0],[ow,0],[ow,oh],[0,oh]])
    return cv2.getPerspectiveTransform(src, dst), px_per_m

def img_to_world(pt, M, px_per_m):
    r = cv2.perspectiveTransform(np.float32([[pt]]).reshape(-1,1,2), M)
    return float(r[0][0][0])/px_per_m, float(r[0][0][1])/px_per_m

def in_polygon(x, y, poly):
    return cv2.pointPolygonTest(np.array(poly, np.int32), (float(x),float(y)), False) >= 0

def assign_lane(cx, cy, lanes):
    if not lanes:
        return 0

    distances = []
    for lane in lanes:
        # Sử dụng measureDist=True để đo khoảng cách chính xác từ tâm xe đến ranh giới làn
        # Trả về số dương nếu xe nằm trong làn, số âm nếu xe nằm ngoài làn
        dist = cv2.pointPolygonTest(np.array(lane, np.int32), (float(cx), float(cy)), True)
        distances.append(dist)

    # np.argmax sẽ tự động tìm ra khoảng cách lớn nhất
    # (Tức là nằm sâu trong làn nhất, hoặc nếu ở ngoài thì nằm gần sát mép làn đó nhất)
    best_lane_idx = np.argmax(distances)

    # Trả về ID làn đường (Lane 1, Lane 2...) thay vì trả về 0
    return int(best_lane_idx + 1)

def draw_count_panel(frame, cum_dir_counts, cum_lane_counts):
    """Displays Cumulative Throughput breakdown."""
    lines = []
    total_all = sum(cum_dir_counts.values())
    lines.append((f"Total Throughput: {total_all}", (255, 255, 255)))

    for d in sorted(cum_dir_counts.keys()):
        count = cum_dir_counts[d]
        dc = DIR_COLORS_BGR.get(d, (200, 200, 200))
        lines.append((f"  {d}: {count}", dc))
        for lane_id, lc_count in sorted(cum_lane_counts[d].items()):
            lc = LANE_COLORS_BGR[(lane_id - 1) % len(LANE_COLORS_BGR)]
            lines.append((f"    L{lane_id}: {lc_count}", lc))

    line_h = 22
    pad_top = 30
    panel_w = 230
    panel_h = pad_top + line_h * len(lines) + 10
    overlay = frame.copy()
    cv2.rectangle(overlay, (5, 5), (panel_w, panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.rectangle(frame, (5, 5), (panel_w, panel_h), (200, 200, 200), 1)
    for i, (text, color) in enumerate(lines):
        cv2.putText(frame, text, (15, pad_top + line_h * i),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

# ── Core tracking ─────────────────────────────────────────────

def run_tracking(config, conf_threshold, iou_threshold, track_thresh, track_buffer, progress_cb=None):
    video_path   = config["video_path"]
    directions   = config["directions"]
    weights_path = config["weights_path"]
    output_dir   = config["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    dir_meta = {}
    for d, data in directions.items():
        M, ppm = build_perspective(data["roi_points"], data["real_w_m"], data["real_h_m"])
        dir_meta[d] = {"M": M, "ppm": ppm, "roi": data["roi_points"], "lanes": data["lanes"]}

    if progress_cb: progress_cb(0.05, "Loading YOLO & Tracker...")
    model   = YOLO(weights_path)
    tracker = sv.ByteTrack(track_activation_threshold=track_thresh,
                           lost_track_buffer=int(track_buffer))

    cap         = cv2.VideoCapture(video_path)
    total       = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps         = cap.get(cv2.CAP_PROP_FPS) or 30
    W, H        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_out   = os.path.join(output_dir, "vehicles-result.mp4")
    writer      = cv2.VideoWriter(video_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W,H))

    # Cumulative Counters
    cum_dir_counts  = {d: 0 for d in directions}
    cum_lane_counts = {d: defaultdict(int) for d in directions}
    counted_tids    = set() # Track unique IDs per direction ROI
    counted_lanes   = set() # Track unique IDs per specific lane

    track_history  = defaultdict(lambda: deque(maxlen=30))
    track_speeds   = defaultdict(list)
    track_max_spd  = defaultdict(float)
    track_cls      = {}
    track_dir      = {}
    csv_rows       = []

    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break

        if progress_cb and frame_idx % 50 == 0:
            progress_cb(0.1 + 0.85*(frame_idx/max(total,1)), f"Frame {frame_idx}/{total}")

        # Inference with specific thresholds
        results    = model(frame, verbose=False, classes=list(VEHICLE_CLASSES.keys()),
                           conf=conf_threshold, iou=iou_threshold)[0]
        detections = sv.Detections.from_ultralytics(results)
        detections = tracker.update_with_detections(detections)

        # Draw Static Overlays
        for d, meta in dir_meta.items():
            dc = DIR_COLORS_BGR[d]
            cv2.polylines(frame, [np.array(meta["roi"], np.int32)], True, dc, 2)
            for li, lane in enumerate(meta["lanes"]):
                lc = LANE_COLORS_BGR[li%len(LANE_COLORS_BGR)]
                cv2.polylines(frame, [np.array(lane, np.int32)], True, lc, 1)

        for i in range(len(detections)):
            if detections.tracker_id is None: continue
            tid  = int(detections.tracker_id[i])
            cls  = int(detections.class_id[i])
            conf = float(detections.confidence[i])
            box  = detections.xyxy[i]
            cx, cy = float((box[0]+box[2])/2), float((box[1]+box[3])/2)

            if tid not in track_cls: track_cls[tid] = VEHICLE_CLASSES.get(cls, "vehicle")

            cur_dir = track_dir.get(tid)
            if cur_dir is None:
                for d, meta in dir_meta.items():
                    if in_polygon(cx, cy, meta["roi"]): cur_dir = d; break
                if cur_dir: track_dir[tid] = cur_dir

            if not cur_dir: continue

            meta    = dir_meta[cur_dir]
            wx, wy  = img_to_world((cx,cy), meta["M"], meta["ppm"])
            history = track_history[tid]
            history.append((wx, wy, frame_idx))

            speed_kmh = 0.0
            if len(history) >= 2:
                wx0, wy0, f0 = history[0]; wx1, wy1, f1 = history[-1]
                dt = (f1-f0)/fps
                if dt > 0:
                    spd = (np.hypot(wx1-wx0, wy1-wy0)/dt)*3.6
                    if spd <= 200: speed_kmh = spd; track_speeds[tid].append(spd); track_max_spd[tid] = max(track_max_spd[tid], spd)

            lane_id = assign_lane(cx, cy, meta["lanes"])

            # --- CUMULATIVE COUNTING LOGIC ---
            if in_polygon(cx, cy, meta["roi"]):
                if (tid, cur_dir) not in counted_tids:
                    cum_dir_counts[cur_dir] += 1
                    counted_tids.add((tid, cur_dir))

                if lane_id > 0 and (tid, cur_dir, lane_id) not in counted_lanes:
                    cum_lane_counts[cur_dir][lane_id] += 1
                    counted_lanes.add((tid, cur_dir, lane_id))

            x1, y1, x2, y2 = map(int, box)
            bbox_color = LANE_COLORS_BGR[(lane_id-1)%6] if lane_id > 0 else DIR_COLORS_BGR[cur_dir]
            cv2.rectangle(frame, (x1, y1), (x2, y2), bbox_color, 2)
            cv2.putText(frame, f"ID{tid} {speed_kmh:.0f}kmh", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, bbox_color, 2)

            csv_rows.append({"frame":frame_idx,"track_id":tid,"direction":cur_dir,"class":track_cls[tid],"img_x":int(cx),"img_y":int(cy),"world_x_m":round(wx,3),"world_y_m":round(wy,3),"speed_kmh":round(speed_kmh,1),"lane":lane_id,"conf":round(conf,3)})

        draw_count_panel(frame, cum_dir_counts, cum_lane_counts)
        writer.write(frame)
        frame_idx += 1

    cap.release(); writer.release()
    # ... (Rest of CSV export logic remains same as original Cell 3) ...
    tracks_csv = os.path.join(output_dir, "vehicle_tracks_xy.csv")
    if csv_rows:
        with open(tracks_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
            w.writeheader(); w.writerows(csv_rows)
    summary_csv = os.path.join(output_dir, "vehicles.csv")
    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["track_id","class","direction","avg_speed_kmh","max_speed_kmh","frame_count"])
        w.writeheader()
        for tid, cls in track_cls.items():
            spds = track_speeds[tid]
            w.writerow({"track_id":tid,"class":cls,"direction":track_dir.get(tid,"unknown"),"avg_speed_kmh":round(sum(spds)/len(spds),1) if spds else 0,"max_speed_kmh":round(track_max_spd[tid],1),"frame_count":len([r for r in csv_rows if r["track_id"]==tid])})
    return {"video": video_out, "tracks": tracks_csv, "summary": summary_csv}

# ── Gradio UI ─────────────────────────────────────────────────

def run_ui(config_path, conf, iou, t_thresh, t_buf, gr_progress=gr.Progress()):
    if not config_path or not os.path.exists(config_path):
        return None, None, None, "❌ Config error"
    try:
        config = json.load(open(config_path))
        outputs = run_tracking(config, conf, iou, t_thresh, t_buf, progress_cb=lambda f,m: gr_progress(f, desc=m))
        return outputs["video"], outputs["tracks"], outputs["summary"], "✅ Done! Cumulative counting complete."
    except Exception: return None, None, None, f"❌ Error: {traceback.format_exc()}"

with gr.Blocks(title="Detection & Throughput", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚦 Detection, Tracking & Throughput (Cell 3 Refactored)")
    with gr.Row():
        with gr.Column():
            cfg_in  = gr.Textbox(label="Config Path", value="/content/output/calib_config.json")
            with gr.Row():
                conf_sl = gr.Slider(0.1, 0.9, value=0.25, step=0.05, label="YOLO Confidence")
                iou_sl  = gr.Slider(0.1, 0.9, value=0.45, step=0.05, label="YOLO IoU")
            with gr.Row():
                t_thresh = gr.Slider(0.1, 0.9, value=0.25, step=0.05, label="ByteTrack Threshold")
                t_buffer = gr.Slider(10, 150, value=30, step=5, label="Track Buffer (Frames)")
            btn_run = gr.Button("▶️ Run Analysis", variant="primary")
            status  = gr.Textbox(label="Status", lines=4)
        with gr.Column():
            out_video   = gr.File(label="vehicles-result.mp4")
            out_tracks  = gr.File(label="vehicle_tracks_xy.csv")
            out_summary = gr.File(label="vehicles.csv")

    btn_run.click(run_ui, [cfg_in, conf_sl, iou_sl, t_thresh, t_buffer], [out_video, out_tracks, out_summary, status])

demo.launch(share=True, quiet=True)

* Running on public URL: https://a3935a6fa5954fc914.gradio.live


### **Cell 4 : Core Logic — Lớp phân tích hiệu suất giao thông (`CVIntersectionAnalyzer`)**

* **Mục đích:** Trở thành "bộ não" tính toán giao thông nội bộ của Notebook, thay thế hoàn toàn cho file mã nguồn `.py` bên ngoài. Cell này chứa các hàm hình học cơ bản và cấu trúc thuật toán phân tích chuyên sâu.
* **Các thành phần logic cốt lõi bên trong:**
  * `point_in_polygon`: Thuật toán kiểm tra một phương tiện có nằm trong vùng làn đường đã vẽ hay không (thuật toán Ray-casting).
  * `calculate_LOS_from_delay`: Hàm phân loại Mức độ phục vụ (Level of Service - LOS) từ hạng A đến F dựa trên thời gian chậm trễ (Delay) theo tiêu chuẩn kỹ thuật giao thông quốc tế.
  * `CVIntersectionAnalyzer`: Lớp xử lý trung tâm chịu trách nhiệm quét qua dữ liệu quỹ đạo từ Cell 3, tính toán độ trễ điều khiển (Control Delay), độ trễ dừng xe (Stopped Delay), số lần dừng, và lưu lượng xe thực tế đi qua ngã tư.

In [ ]:
import pandas as pd
import numpy as np
import json
import csv
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict

# ============================================================================
# CONFIGURATION
# ============================================================================

STOPPED_SPEED_THRESHOLD = 0.5  # m/s (< 1.8 km/h considered stopped)
QUEUE_SPEED_THRESHOLD = 1.0    # m/s
FREE_FLOW_PERCENTILE = 85
MEASUREMENT_START_TIME = 0.0
MIN_TRACK_POINTS = 3
MIN_FRAMES_FOR_COUNTING = 10
MIN_DETECTION_TIME = 0.5

# ============================================================================
# GEOMETRY FUNCTIONS
# ============================================================================

def point_in_polygon(x, y, poly_points):
    n = len(poly_points)
    inside = False
    p1x, p1y = poly_points[0]
    for i in range(1, n + 1):
        p2x, p2y = poly_points[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

def assign_lane_to_point(img_x, img_y, lane_polygons):
    for lane_id, polygon in lane_polygons.items():
        if point_in_polygon(img_x, img_y, polygon):
            return lane_id
    return None

def calculate_LOS_from_delay(control_delay):
    # Modified thresholds to make LOS 'worse' for a given delay
    if control_delay <= 5: return 'A'
    elif control_delay <= 10: return 'B'
    elif control_delay <= 20: return 'C'
    elif control_delay <= 30: return 'D'
    elif control_delay <= 40: return 'E'
    else: return 'F'

@dataclass
class VehicleMetrics:
    vehicle_id: int
    lane_id: int
    first_seen: float
    last_seen: float
    total_time: float
    total_distance: float
    avg_speed: float
    max_speed: float
    stopped_delay: float
    control_delay: float
    queue_time: float
    free_flow_time: float
    completed: bool
    num_stops: int
    trajectory_points: int

@dataclass
class LaneGroupMetrics:
    lane_id: int
    throughput: int
    n_vehicles_completed: int
    n_vehicles_total: int
    avg_control_delay: float
    avg_stopped_delay: float
    avg_speed: float
    measured_flow: float
    los: str
    completion_rate: float

def convert_to_python_types(obj):
    if isinstance(obj, (np.int64, np.int32, np.int16, np.int8)):
        return int(obj)
    elif isinstance(obj, (np.float64, np.float32, np.float16)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {convert_to_python_types(k): convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_to_python_types(item) for item in obj)
    return obj

# ============================================================================
# MAIN ANALYZER CLASS
# ============================================================================

class CVIntersectionAnalyzer:
    def __init__(self, trajectory_file: str, free_flow_speed_kmh: float = 50.0, verbose: bool = True):
        self.trajectory_file = trajectory_file
        self.free_flow_speed_mps = free_flow_speed_kmh / 3.6
        self.verbose = verbose

        self.df = pd.read_csv(trajectory_file)

        if 'lane_id' in self.df.columns:
            self.df = self.df[self.df['lane_id'] >= 0].copy()

        self._preprocess_data()
        self.vehicle_metrics: Dict[int, VehicleMetrics] = {}
        self.lane_metrics: Dict[int, LaneGroupMetrics] = {}
        self.global_metrics: Dict = {}

    def _preprocess_data(self):
        self.df = self.df.sort_values(['vehicle_id', 'time_s']).reset_index(drop=True)
        self.df['dx'] = self.df.groupby('vehicle_id')['x_m'].diff()
        self.df['dy'] = self.df.groupby('vehicle_id')['y_m'].diff()
        self.df['dt'] = self.df.groupby('vehicle_id')['time_s'].diff()

        self.df['distance'] = np.sqrt(self.df['dx']**2 + self.df['dy']**2)
        self.df['speed_ms'] = self.df['distance'] / self.df['dt']
        self.df['speed_kmh'] = self.df['speed_ms'] * 3.6

        self.df['speed_ms'] = self.df['speed_ms'].fillna(0)
        self.df['speed_kmh'] = self.df['speed_kmh'].fillna(0)

        max_realistic_speed = 30.0
        self.df.loc[self.df['speed_ms'] > max_realistic_speed, 'speed_ms'] = np.nan
        self.df.loc[self.df['speed_kmh'] > max_realistic_speed * 3.6, 'speed_kmh'] = np.nan

        self.df['speed_ms'] = self.df.groupby('vehicle_id')['speed_ms'].ffill()
        self.df['speed_kmh'] = self.df.groupby('vehicle_id')['speed_kmh'].ffill()

        self.df['is_stopped'] = self.df['speed_ms'] < STOPPED_SPEED_THRESHOLD
        self.df['is_queued'] = self.df['speed_ms'] < QUEUE_SPEED_THRESHOLD

    def analyze(self, print_results=True):
        self._analyze_vehicles()
        self._analyze_lane_groups()
        self._calculate_global_metrics()
        return self.global_metrics

    def _analyze_vehicles(self):
        track_lengths = self.df.groupby('vehicle_id').size()
        valid_tracks = track_lengths[track_lengths >= MIN_TRACK_POINTS].index

        for vehicle_id in valid_tracks:
            vehicle_data = self.df[self.df['vehicle_id'] == vehicle_id].copy()
            if len(vehicle_data) < MIN_FRAMES_FOR_COUNTING: continue

            first_seen = vehicle_data['time_s'].min()
            last_seen = vehicle_data['time_s'].max()
            total_time = last_seen - first_seen
            if total_time < MIN_DETECTION_TIME: continue

            total_distance = vehicle_data['distance'].sum()
            avg_speed = vehicle_data['speed_ms'].mean()
            max_speed = vehicle_data['speed_ms'].max()

            stopped_delay = vehicle_data.loc[vehicle_data['is_stopped'], 'dt'].sum()

            if total_distance > 0 and self.free_flow_speed_mps > 0:
                free_flow_time = total_distance / self.free_flow_speed_mps
            else:
                free_flow_time = total_time

            control_delay = max(0, total_time - free_flow_time)
            queue_time = vehicle_data.loc[vehicle_data['is_queued'], 'dt'].sum()

            is_stopped_arr = vehicle_data['is_stopped'].values
            num_stops = np.sum(np.diff(is_stopped_arr.astype(int)) > 0)
            completed = first_seen >= MEASUREMENT_START_TIME

            lane_id = int(vehicle_data['lane_id'].mode()[0]) if 'lane_id' in vehicle_data.columns else 0

            self.vehicle_metrics[vehicle_id] = VehicleMetrics(
                vehicle_id=vehicle_id, lane_id=lane_id, first_seen=first_seen, last_seen=last_seen,
                total_time=total_time, total_distance=total_distance, avg_speed=avg_speed, max_speed=max_speed,
                stopped_delay=stopped_delay, control_delay=control_delay, queue_time=queue_time,
                free_flow_time=free_flow_time, completed=completed, num_stops=num_stops, trajectory_points=len(vehicle_data)
            )

    def _analyze_lane_groups(self):
        measurement_duration = self.df['time_s'].max() - MEASUREMENT_START_TIME
        lanes = [lid for lid in self.df['lane_id'].unique() if lid >= 0]

        for lane_id in lanes:
            lane_vehicles = [m for m in self.vehicle_metrics.values() if m.lane_id == lane_id]
            if not lane_vehicles: continue

            n_vehicles = len(lane_vehicles)
            completed = [v for v in lane_vehicles if v.completed]
            n_completed = len(completed)
            if n_completed == 0: continue

            avg_control_delay = np.mean([v.control_delay for v in lane_vehicles])
            avg_stopped_delay = np.mean([v.stopped_delay for v in lane_vehicles])
            avg_speed = np.mean([v.avg_speed for v in completed])

            measured_flow = (n_vehicles / measurement_duration) * 3600 if measurement_duration > 0 else 0
            los = calculate_LOS_from_delay(avg_control_delay)
            completion_rate = n_completed / n_vehicles if n_vehicles > 0 else 0

            self.lane_metrics[lane_id] = LaneGroupMetrics(
                lane_id=lane_id, throughput=n_vehicles, n_vehicles_completed=n_completed,
                n_vehicles_total=n_vehicles, avg_control_delay=avg_control_delay,
                avg_stopped_delay=avg_stopped_delay, avg_speed=avg_speed,
                measured_flow=measured_flow, los=los, completion_rate=completion_rate
            )

    def _calculate_global_metrics(self):
        completed = [v for v in self.vehicle_metrics.values() if v.completed]
        all_vehicles = list(self.vehicle_metrics.values())
        measurement_duration = self.df['time_s'].max() - MEASUREMENT_START_TIME

        avg_control_delay = np.mean([v.control_delay for v in all_vehicles]) if all_vehicles else 0
        avg_stopped_delay = np.mean([v.stopped_delay for v in all_vehicles]) if all_vehicles else 0

        self.global_metrics = {
            'n_vehicles_total': len(all_vehicles),
            'n_vehicles_completed': len(completed),
            'completion_rate': len(completed) / len(all_vehicles) if all_vehicles else 0,
            'avg_control_delay': avg_control_delay,
            'avg_stopped_delay': avg_stopped_delay,
            'avg_speed_ms': np.mean([v.avg_speed for v in completed]) if completed else 0,
            'avg_speed_kmh': np.mean([v.avg_speed for v in completed]) * 3.6 if completed else 0,
            'total_throughput': len(completed),
            'throughput_per_hour': (len(completed) / measurement_duration * 3600) if measurement_duration > 0 else 0,
            'measurement_duration': measurement_duration,
            'overall_los': calculate_LOS_from_delay(avg_control_delay),
            'lane_metrics': {lid: vars(lm) for lid, lm in self.lane_metrics.items()}
        }

    def export_csv(self, output_path: str, include_delay: bool = False, include_los: bool = False):
        total_duration = self.df['time_s'].max() - self.df['time_s'].min()
        num_minutes = int(np.ceil(total_duration / 60))
        lane_ids = sorted([lid for lid in self.df['lane_id'].unique() if lid >= 0])
        counted_vehicles_per_lane = {lane_id: set() for lane_id in lane_ids}
        rows = []

        for minute_idx in range(num_minutes):
            minute = minute_idx + 1
            start_time = minute_idx * 60
            end_time = (minute_idx + 1) * 60
            minute_data = self.df[(self.df['time_s'] >= start_time) & (self.df['time_s'] < end_time)].copy()

            if len(minute_data) == 0:
                for lane_id in lane_ids:
                    rows.append({'Minute': minute, 'lane_id': lane_id, 'n_vehicles': 0, 'avg_speed_kmh': 0.0})
                continue

            for lane_id in lane_ids:
                lane_minute_data = minute_data[minute_data['lane_id'] == lane_id]
                if len(lane_minute_data) == 0:
                    rows.append({'Minute': minute, 'lane_id': lane_id, 'n_vehicles': 0, 'avg_speed_kmh': 0.0})
                    continue

                vehicles_this_minute = set(lane_minute_data['vehicle_id'].unique())
                new_vehicles = vehicles_this_minute - counted_vehicles_per_lane[lane_id]
                n_new_vehicles = len(new_vehicles)
                counted_vehicles_per_lane[lane_id].update(new_vehicles)

                if 'speed_kmh' in lane_minute_data.columns:
                    avg_speed = lane_minute_data['speed_kmh'].mean()
                else:
                    avg_speed = lane_minute_data['speed_ms'].mean() * 3.6 if 'speed_ms' in lane_minute_data.columns else 0.0

                row_data = {'Minute': minute, 'lane_id': lane_id, 'n_vehicles': n_new_vehicles, 'avg_speed_kmh': round(avg_speed, 2)}

                if include_delay or include_los:
                    new_vehicles_data = lane_minute_data[lane_minute_data['vehicle_id'].isin(new_vehicles)]
                    if len(new_vehicles_data) > 0 and n_new_vehicles > 0:
                        delays = []
                        for vid in new_vehicles:
                            veh_data = new_vehicles_data[new_vehicles_data['vehicle_id'] == vid]
                            if len(veh_data) > 1:
                                if 'distance' in veh_data.columns:
                                    distance = veh_data['distance'].sum()
                                else:
                                    coords = veh_data[['x_m', 'y_m']].values
                                    if len(coords) > 1:
                                        distance = np.sum(np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1)))
                                    else:
                                        distance = 0
                                actual_time = veh_data['time_s'].max() - veh_data['time_s'].min()
                                if distance > 0 and self.free_flow_speed_mps > 0:
                                    free_flow_time = distance / self.free_flow_speed_mps
                                    delay = max(0, actual_time - free_flow_time)
                                    delays.append(delay)

                        avg_delay = np.mean(delays) if delays else 0.0
                        if include_delay: row_data['avg_delay_s'] = round(avg_delay, 2)
                        if include_los: row_data['los'] = calculate_LOS_from_delay(avg_delay)
                    else:
                        if include_delay: row_data['avg_delay_s'] = 0.0
                        if include_los: row_data['los'] = 'A'

                rows.append(row_data)

        df = pd.DataFrame(rows)
        df.to_csv(output_path, index=False)


### **Cell 5: Analytics — Phân tích dữ liệu theo hướng**

* **Mục đích:** Xử lý file dữ liệu quỹ đạo (`vehicle_tracks_xy.csv`) để trích xuất các chỉ số đánh giá hiệu suất giao thông theo tiêu chuẩn.
* **Quá trình xử lý logic:**
  * Nhập các tham số đầu vào qua Gradio như Tốc độ dòng tự do (Free-flow speed - km/h).
  * Sử dụng một module tùy chỉnh (`CVIntersectionAnalyzer`) để đánh giá dữ liệu của từng hướng (North, South, East, West).
  * Tính toán các chỉ số vĩ mô: Xếp hạng Mức độ phục vụ (LOS từ A đến F), Độ trễ điều khiển (Control Delay), Độ trễ dừng (Stopped Delay), và Lưu lượng thông qua (Throughput).


* **Đầu ra (Output):**
  * `analytics.csv`: Dữ liệu phân tích chi tiết theo chuỗi thời gian (từng phút).
  * `summary.json`: Dữ liệu tổng hợp gọn nhẹ chứa các chỉ số KPI hiệu suất cho toàn bộ ngã tư và chia nhỏ theo từng làn.

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import json, os, sys, traceback

DIRECTIONS = ["North", "South", "East", "West"]

def classify_movement(start_x, start_y, end_x, end_y, direction):
    """
    Classifies vehicle movement using PIXEL coordinates (img_x, img_y).
    Based on the provided camera view (oblique/diagonal):
    - North: Moving Top-Left to Bottom-Right (~45 deg)
    - South: Moving Bottom-Right to Top-Left (~-135 deg)
    - East:  Moving Top-Right to Bottom-Left (~135 deg)
    - West:  Moving Bottom-Left to Top-Right (~-45 deg)
    """
    dx = end_x - start_x
    dy = end_y - start_y
    dist = np.hypot(dx, dy)

    # Requirement: Vehicle must move at least 50 pixels to classify
    if dist < 50.0:
        return "Stationary"

    # Calculate pixel-space angle (-180 to 180)
    angle = np.degrees(np.arctan2(dy, dx))

    # Alignment map based on the specific diagonal intersection view
    angle_map = {
        "North": 45,
        "South": -135,
        "East":  135,
        "West":  -45
    }

    if direction not in angle_map: return "Straight"

    target_angle = angle_map[direction]

    # Normalized angular difference [-180, 180]
    diff = (angle - target_angle + 180) % 360 - 180

    # Thresholds (30 deg tolerance for straight, 35-120 deg for turns)
    if abs(diff) < 30:
        return "Straight"
    elif -120 < diff < -35:
        return "Left Turn"
    elif 35 < diff < 120:
        return "Right Turn"

    return "Straight"

def prep_df_for_analyzer(df):
    df = df.rename(columns={"track_id":"vehicle_id","world_x_m":"x_m",
                             "world_y_m":"y_m","lane":"lane_id"})
    if "time_s" not in df.columns:
        df["time_s"] = df["frame"] / 30.0
    return df

def run_analytics(tracks_path, out_dir, free_flow, inc_delay, inc_los, gr_progress=gr.Progress()):
    if not tracks_path or not os.path.exists(tracks_path):
        return None, None, None, "❌ Không tìm thấy vehicle_tracks_xy.csv"

    os.makedirs(out_dir, exist_ok=True)
    df_all = pd.read_csv(tracks_path)

    all_summaries = {}
    analytics_csv = os.path.join(out_dir, "analytics.csv")
    summary_json  = os.path.join(out_dir, "summary.json")
    tmc_csv       = os.path.join(out_dir, "tmc.csv")
    all_analytics = []
    tmc_results   = []

    directions = df_all["direction"].unique() if "direction" in df_all.columns else ["all"]

    for i, d in enumerate(directions):
        gr_progress((i+1)/len(directions), desc=f"Phân tích {d}...")

        df_dir = df_all[df_all["direction" ] == d].copy() if d != "all" else df_all.copy()

        # --- TMC Logic (Updated to use pixel coordinates for stability) ---
        for tid in df_dir['track_id'].unique():
            t_data = df_dir[df_dir['track_id'] == tid].sort_values('frame')
            if len(t_data) < 10: continue
            start, end = t_data.iloc[0], t_data.iloc[-1]
            # USE PIXEL COORDINATES (img_x, img_y) for classify_movement
            mov = classify_movement(start['img_x'], start['img_y'], end['img_x'], end['img_y'], d)
            tmc_results.append({"track_id": tid, "direction": d, "class": start['class'], "movement": mov})

        # --- Performance Logic ---
        df_prep = prep_df_for_analyzer(df_dir)
        tmp_csv = os.path.join(out_dir, f"_tmp_{d}.csv")
        df_prep.to_csv(tmp_csv, index=False)

        try:
            analyzer = CVIntersectionAnalyzer(trajectory_file=tmp_csv, free_flow_speed_kmh=float(free_flow), verbose=False)
            metrics = analyzer.analyze(print_results=False)
            analyzer.export_csv(os.path.join(out_dir, f"analytics_{d}.csv"), include_delay=inc_delay, include_los=inc_los)
            df_a = pd.read_csv(os.path.join(out_dir, f"analytics_{d}.csv"))
            df_a.insert(0, "direction", d)
            all_analytics.append(df_a)
            all_summaries[d] = {k: v for k, v in metrics.items() if k != "lane_metrics"}
            all_summaries[d]["lane_metrics"] = {str(k): v for k,v in metrics.get("lane_metrics",{}).items()}
        except Exception: all_summaries[d] = {"error": traceback.format_exc()}
        finally:
            if os.path.exists(tmp_csv): os.remove(tmp_csv)

    if all_analytics: pd.concat(all_analytics, ignore_index=True).to_csv(analytics_csv, index=False)
    if tmc_results: pd.DataFrame(tmc_results).to_csv(tmc_csv, index=False)
    with open(summary_json, "w") as f: json.dump(all_summaries, f, indent=2, default=str)

    status = f"✅ Xong! Đã xuất analytics.csv, summary.json và tmc.csv"
    return analytics_csv, summary_json, tmc_csv, status

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 Analytics & TMC")
    with gr.Row():
        with gr.Column():
            tracks_in = gr.Textbox(label="Tracks CSV", value="/content/output/vehicle_tracks_xy.csv")
            out_dir_in = gr.Textbox(label="Output folder", value="/content/output")
            free_flow = gr.Slider(20, 100, value=50, label="Free-flow speed")
            btn = gr.Button("▶️ Chạy Analytics", variant="primary")
            status = gr.Textbox(label="Kết quả", lines=5)
        with gr.Column():
            out_an = gr.File(label="analytics.csv")
            out_sm = gr.File(label="summary.json")
            out_tmc = gr.File(label="tmc.csv (Turning Movement)")

    btn.click(run_analytics, [tracks_in, out_dir_in, free_flow, gr.State(True), gr.State(True)], [out_an, out_sm, out_tmc, status])

demo.launch(share=True, quiet=True)

* Running on public URL: https://cec337bb9c3f8c2ca8.gradio.live


### **Cell 6: Visualization Dashboard — Bảng điều khiển trực quan**

* **Mục đích:** Đọc kết quả từ Cell 4 và render thành các biểu đồ báo cáo dễ hiểu mà không cần các phần mềm BI bên ngoài, sử dụng trực tiếp Matplotlib.
* **Giao diện & Chức năng:**
  * Cung cấp một Dropdown list để chọn Hướng cần xem báo cáo.
  * **Plot 1 (Throughput theo phút):** Biểu đồ đường (Line chart) thể hiện lưu lượng xe mới đi vào ngã tư theo từng phút, chia theo từng làn.
  * **Plot 2 (Tốc độ theo phút):** Biểu đồ đường thể hiện sự biến thiên tốc độ trung bình theo thời gian thực.
  * **Plot 3 (LOS theo làn):** Biểu đồ cột (Bar chart) hiển thị Độ trễ điều khiển (Control Delay). Các cột được gán nhãn dán và tô màu theo tiêu chuẩn LOS (Màu xanh cho A/B, Đỏ cho E/F).
  * **Plot 4 (Tổng kết):** Dưới dạng một bảng Card (Bảng số liệu tĩnh) tóm tắt nhanh các chỉ số cốt lõi: Tổng xe, Tốc độ TB, Thời gian delay, LOS tổng thể của toàn bộ hướng được chọn.

In [ ]:
# ============================================================
# CELL 6 — Visualization Dashboard
# Đọc summary.json + analytics.csv từ Cell 4 và vẽ biểu đồ
# Không cần thêm thư viện — dùng matplotlib có sẵn trong Colab
# ============================================================

import gradio as gr
import pandas as pd
import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

LOS_COLORS = {"A":"#2ecc71","B":"#27ae60","C":"#f1c40f","D":"#e67e22","E":"#e74c3c","F":"#8e44ad"}


def load_data(summary_path, analytics_path):
    if not os.path.exists(summary_path):
        return None, None, f"❌ Không tìm thấy {summary_path}"
    if not os.path.exists(analytics_path):
        return None, None, f"❌ Không tìm thấy {analytics_path}"
    with open(summary_path) as f:
        summary = json.load(f)
    df = pd.read_csv(analytics_path)
    return summary, df, "✅ Đã load dữ liệu"


def make_dashboard(summary_path, analytics_path, selected_dir):
    summary_all, df_full, msg = load_data(summary_path, analytics_path)
    if summary_all is None:
        return None, None, None, None, msg

    # Lấy summary của direction được chọn
    if selected_dir and selected_dir in summary_all:
        summary = summary_all[selected_dir]
        df = df_full[df_full["direction"] == selected_dir].copy() if "direction" in df_full.columns else df_full
    else:
        # Fallback: lấy direction đầu tiên có data
        first_dir = next(iter(summary_all))
        summary = summary_all[first_dir]
        df = df_full[df_full["direction"] == first_dir].copy() if "direction" in df_full.columns else df_full

    lane_ids = sorted(df["lane_id"].unique())
    colors   = plt.cm.tab10(np.linspace(0, 1, max(len(lane_ids), 1)))

    # ── Plot 1: Throughput per minute per lane ────────────────
    fig1, ax1 = plt.subplots(figsize=(9, 4))
    for i, lid in enumerate(lane_ids):
        sub = df[df["lane_id"] == lid]
        ax1.plot(sub["Minute"], sub["n_vehicles"], marker="o", markersize=3,
                 label=f"Lane {lid}", color=colors[i])
    ax1.set_title(f"Số xe mới mỗi phút theo lane ({selected_dir})", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Phút"); ax1.set_ylabel("Xe mới")
    ax1.legend(); ax1.grid(alpha=0.3)
    fig1.tight_layout()

    # ── Plot 2: Tốc độ TB theo phút per lane ─────────────────
    fig2, ax2 = plt.subplots(figsize=(9, 4))
    for i, lid in enumerate(lane_ids):
        sub = df[df["lane_id"] == lid]
        ax2.plot(sub["Minute"], sub["avg_speed_kmh"], marker="s", markersize=3,
                 label=f"Lane {lid}", color=colors[i])
    ax2.set_title(f"Tốc độ trung bình theo phút ({selected_dir})", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Phút"); ax2.set_ylabel("km/h")
    ax2.legend(); ax2.grid(alpha=0.3)
    fig2.tight_layout()

    # ── Plot 3: LOS per lane (bar) ────────────────────────────
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    lane_data = summary.get("lane_metrics", {})
    if lane_data:
        lids   = [str(k) for k in sorted(lane_data.keys(), key=lambda x: int(str(x)))]
        delays = [lane_data[k]["avg_control_delay"] for k in sorted(lane_data.keys(), key=lambda x: int(str(x)))]
        los_list = [lane_data[k]["los"] for k in sorted(lane_data.keys(), key=lambda x: int(str(x)))]
        bar_colors = [LOS_COLORS.get(l, "#95a5a6") for l in los_list]
        bars = ax3.bar([f"Lane {l}" for l in lids], delays, color=bar_colors, edgecolor="white", linewidth=1.2)
        for bar, los in zip(bars, los_list):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f"LOS {los}", ha="center", va="bottom", fontsize=10, fontweight="bold")
        patches = [mpatches.Patch(color=v, label=f"LOS {k}") for k, v in LOS_COLORS.items()]
        ax3.legend(handles=patches, fontsize=8, ncol=3, loc="upper right")
    ax3.set_title(f"Control Delay & LOS theo lane ({selected_dir})", fontsize=13, fontweight="bold")
    ax3.set_ylabel("Avg control delay (s/veh)"); ax3.grid(axis="y", alpha=0.3)
    fig3.tight_layout()

    # ── Plot 4: Global summary card ───────────────────────────
    fig4, ax4 = plt.subplots(figsize=(6, 4))
    ax4.axis("off")
    overall_los = summary.get("overall_los", "?")
    rows = [
        ["Tổng xe",          f"{summary.get('n_vehicles_total', 0)}"],
        ["Throughput",       f"{summary.get('throughput_per_hour', 0):.0f} veh/h"],
        ["Tốc độ TB",        f"{summary.get('avg_speed_kmh', 0):.1f} km/h"],
        ["Control delay TB", f"{summary.get('avg_control_delay', 0):.1f} s/veh"],
        ["Stopped delay TB", f"{summary.get('avg_stopped_delay', 0):.1f} s/veh"],
        ["LOS tổng thể",     overall_los],
        ["Thời gian đo",     f"{summary.get('measurement_duration', 0):.0f} s"],
    ]
    tbl = ax4.table(cellText=rows, colLabels=["Chỉ số", "Giá trị"],
                    cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(11)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white", fontweight="bold")
        elif r > 0 and rows[r-1][0] == "LOS tổng thể":
            cell.set_facecolor(LOS_COLORS.get(overall_los, "#95a5a6"))
            cell.set_text_props(fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#ecf0f1")
    ax4.set_title(f"📋 Tổng kết ({selected_dir})", fontsize=13, fontweight="bold", pad=10)
    fig4.tight_layout()

    return fig1, fig2, fig3, fig4, "✅ Dashboard sẵn sàng!"


# ── UI ────────────────────────────────────────────────────────

with gr.Blocks(title="Dashboard", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📈 Traffic Analysis — Bước 4: Visualization Dashboard")

    with gr.Row():
        summary_in   = gr.Textbox(label="summary.json",   value="/content/output/summary.json")
        analytics_in = gr.Textbox(label="analytics.csv",  value="/content/output/analytics.csv")
        dir_select   = gr.Dropdown(label="Direction", choices=["North","South","East","West"], value="North")
        btn = gr.Button("📊 Vẽ biểu đồ", variant="primary")

    status = gr.Textbox(label="Trạng thái", interactive=False)

    with gr.Row():
        plot_throughput = gr.Plot(label="Throughput theo phút")
        plot_speed      = gr.Plot(label="Tốc độ theo phút")
    with gr.Row():
        plot_los     = gr.Plot(label="LOS theo lane")
        plot_summary = gr.Plot(label="Tổng kết")

    btn.click(
        make_dashboard,
        inputs=[summary_in, analytics_in, dir_select],
        outputs=[plot_throughput, plot_speed, plot_los, plot_summary, status],
    )

demo.launch(share=True, quiet=True)

* Running on public URL: https://c3fa94ed01a693014c.gradio.live


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
